In [ ]:
from textacy import preprocessing
import trafilatura
from bs4 import BeautifulSoup as bs
import pandas as pd
import pickle
import os
import requests

In [ ]:
url = "https://www.viaggiavventurenelmondo.it"

if os.path.isfile("lista_URLs_viaggiavventure.pkl"):
    with open("lista_URLs_viaggiavventure.pkl", "rb") as fp:
        list_viaggi = pickle.load(fp)
else:
    list_viaggi = []
    for i in range(0, 10000):
        try:
            url_loc = url + f"/viaggi/{i:04}"
            downloaded = trafilatura.fetch_url(url_loc)
            soup = bs(downloaded, 'lxml')
            soup.text
            list_viaggi.append(url_loc)
            print(i)
            print(url_loc)
        except:
            continue
        
    with open("lista_URLs_viaggiavventure.pkl", "wb") as fp:
        pickle.dump(list_viaggi, fp)

In [ ]:
df = pd.DataFrame(columns = ["Country", "Nome_Viaggio", "Link", "Descrizione Viaggio", "Mood Viaggio", "Itinerario", "Sottotitolo_Viaggio"])
for url_viaggio in list_viaggi:
    response = requests.get(url_viaggio)
    response.encoding = 'utf-8'
    soup = bs(response.text, 'html.parser')
    str_soup = str(soup)
    
    info_viaggio = {}
    title = soup.find(attrs = {"class": "base__header-name"}).text.strip()
    subtitle = soup.find(attrs = {"class": "description"}).text.strip()
    flags = soup.find(attrs = {"class": "base__header-flags"})
    flags_imgs = flags.find_all("img")
    countries_regions = set()
    for img in flags_imgs:
        countries_regions.add(img['title'])
        
    l = soup.find("span", attrs={"class": "base__header-subtext"})
    links = l.find_all('a')
    trip_type = links[0].text
    countries_regions.add(links[1].text)
    countries_regions = list(countries_regions)
    
    
    duration = soup.find(attrs = {"class": "duration"}).text.strip()
    duration_measure = str_soup.split(f'<div class="title">DURATA</div>\n<div class="value"><span class="duration">{duration}</span>')[1].split('</div>')[0].strip()
    duration = duration + " " + duration_measure
    period = str_soup.split('<div class="title">PERIODO</div>\n<div class="value">')[1].split('</div>')[0].strip()
    difficulty = str_soup.split('<div class="title">DIFFICOLTÀ</div>\n<div class="value">')[1].split('</div>')[0].strip()
    
    info_viaggio['Durata'] = duration
    info_viaggio['Periodo'] = period
    info_viaggio['Difficoltà'] = difficulty
    info_viaggio['Type'] = trip_type
    
    text_snippets = soup.find_all(attrs={"class": "text01__main text01__simple snippets__default"})
    description_list = []
    for snip in text_snippets:
        description_list.append(snip.text.strip())
    description = " ".join(description_list)
    
    itinerary_dict = {}
    itinerary = soup.find(attrs = {"class": "itinerary01__main snippets__narrow"})
    if itinerary:
        itinerary_days = itinerary.find_all(attrs = {"class": "row"})
        for itiner_day in itinerary_days:
            day =  itiner_day.find(attrs = {"class": "counter"}).text
            places =  itiner_day.find(attrs = {"class": "label"}).text
            itinerary_dict[day] = places
        
    row = pd.DataFrame.from_dict({"Country": [countries_regions],
                         "Nome_Viaggio": [title],
                          "Link": [url_viaggio],
                          "Descrizione Viaggio": [description],
                          "Mood Viaggio": [info_viaggio],
                         "Itinerario": [itinerary_dict],
                         "Sottotitolo_Viaggio": [subtitle]})
    df = pd.concat([df, row], ignore_index=True)
    

In [ ]:
len(list_viaggi)

In [ ]:
df.shape

In [ ]:
df

In [ ]:
df.to_pickle("df_viaggi_viaggiavventure.pkl")